# Bronze Layer Notebook

Load raw CSV files into Delta Lake Bronze layer with minimal transformation.

In [1]:
from datetime import datetime, timezone
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.functions import lit

In [2]:
MINIO_ENDPOINT = "http://localhost:9010"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin123"
MINIO_BUCKET = "crypto-warehouse"

import sys
import subprocess

# Keep Spark and Delta versions compatible
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pyspark==4.0.0", "delta-spark==4.0.0"])

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Stop any existing session so new dependency/config set is applied
try:
    spark.stop()
except Exception:
    pass

builder = (
    SparkSession.builder.appName("DataWarehouse-ETL")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    # Delta Lake (explicitly required)
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # MinIO (S3A)
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
)

extra_packages = [
    "org.apache.hadoop:hadoop-aws:3.4.1",
    "software.amazon.awssdk:bundle:2.31.58",
]

spark = configure_spark_with_delta_pip(builder, extra_packages=extra_packages).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("spark.sql.extensions =", spark.conf.get("spark.sql.extensions", "<missing>"))
print("spark.sql.catalog.spark_catalog =", spark.conf.get("spark.sql.catalog.spark_catalog", "<missing>"))
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/23 12:08:11 WARN Utils: Your hostname, bnguyen-TX-Air-FA401KM-FA401KM, resolves to a loopback address: 127.0.1.1; using 192.168.1.9 instead (on interface wlp99s0)
26/04/23 12:08:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/bnguyen/Desktop/finnhub-streaming-pipeline/venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/bnguyen/.ivy2.5.2/cache
The jars for the packages stored in: /home/bnguyen/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
software.amazon.awssdk#bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-755de2c7-7c40-44a9-acce-87abc21a07f8;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.0 in central
	found io.delta#delt

spark.sql.extensions = io.delta.sql.DeltaSparkSessionExtension
spark.sql.catalog.spark_catalog = org.apache.spark.sql.delta.catalog.DeltaCatalog


In [3]:
# Local datasets path + direct MinIO Delta destination
REPO_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATASETS_DIR = REPO_ROOT / "datasets"
BASE_URI = f"s3a://{MINIO_BUCKET}"
BRONZE_PATH = f"{BASE_URI}/bronze"

print("REPO_ROOT:", REPO_ROOT)
print("DATASETS_DIR:", DATASETS_DIR)
print("MINIO_ENDPOINT:", MINIO_ENDPOINT)
print("BRONZE_PATH:", BRONZE_PATH)

REPO_ROOT: /home/bnguyen/Desktop/finnhub-streaming-pipeline
DATASETS_DIR: /home/bnguyen/Desktop/finnhub-streaming-pipeline/datasets
MINIO_ENDPOINT: http://localhost:9010
BRONZE_PATH: s3a://crypto-warehouse/bronze


In [6]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

schema = StructType([
    StructField("Date", StringType(), True),
    StructField("Open", DoubleType(), True),
    StructField("High", DoubleType(), True),
    StructField("Low", DoubleType(), True),
    StructField("Close", DoubleType(), True),
    StructField("Volume", LongType(), True),
    StructField("Market Cap", DoubleType(), True)
])

csv_files = {
    "BITCOIN24.csv": "BTC",
    "ETHEREUM.csv": "ETH",
    "LITECOIN24.csv": "LTC",
}

for csv_name, symbol in csv_files.items():
    csv_path = DATASETS_DIR / csv_name
    if not csv_path.exists():
        print(f"Skip missing file: {csv_path}")
        continue

    raw_df = (
        spark.read
        .option("header", "true")
        .schema(schema)   
        .csv(str(csv_path))
    )

    bronze_df = (
        raw_df
        .withColumn("asset_symbol", lit(symbol))
        .withColumn("source_file", lit(csv_name))
        .withColumn("ingestion_ts", lit(datetime.now(timezone.utc).isoformat()))
    )

    (
        bronze_df.write
        .format("delta")
        .mode("append")
        # Allow source column names like "Market Cap" by enabling Delta column mapping.
        .option("delta.columnMapping.mode", "name")
        .option("delta.minReaderVersion", "2")
        .option("delta.minWriterVersion", "5")
        .save(BRONZE_PATH)
    )

    print(f"Loaded {csv_name} ({symbol}): {bronze_df.count()} rows")

Loaded BITCOIN24.csv (BTC): 2250 rows


26/04/23 12:08:56 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Date, Open, High, Low, Close, Adj Close, Volume
 Schema: Date, Open, High, Low, Close, Volume, Market Cap
Expected: Volume but found: Adj Close
CSV file: file:///home/bnguyen/Desktop/finnhub-streaming-pipeline/datasets/ETHEREUM.csv


Loaded ETHEREUM.csv (ETH): 2404 rows


26/04/23 12:08:56 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Date, Open, High, Low, Close, Adj Close, Volume
 Schema: Date, Open, High, Low, Close, Volume, Market Cap
Expected: Volume but found: Adj Close
CSV file: file:///home/bnguyen/Desktop/finnhub-streaming-pipeline/datasets/LITECOIN24.csv


Loaded LITECOIN24.csv (LTC): 2404 rows


In [7]:
bronze = spark.read.format("delta").load(BRONZE_PATH)
print("Total Bronze rows:", bronze.count())
bronze.groupBy("asset_symbol").count().orderBy("asset_symbol").show()
bronze.printSchema()
bronze.show(10, truncate=False)

Total Bronze rows: 7058
+------------+-----+
|asset_symbol|count|
+------------+-----+
|         BTC| 2250|
|         ETH| 2404|
|         LTC| 2404|
+------------+-----+

root
 |-- Date: string (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: long (nullable = true)
 |-- Market Cap: double (nullable = true)
 |-- asset_symbol: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_ts: string (nullable = true)

+----------+----------+----------+----------+----------+------+-------------+------------+--------------+--------------------------------+
|Date      |Open      |High      |Low       |Close     |Volume|Market Cap   |asset_symbol|source_file   |ingestion_ts                    |
+----------+----------+----------+----------+----------+------+-------------+------------+--------------+--------------------------------+
|01-01-2018|231.